# ⭐ Day 79: Final Project 1 - End-to-End Customer Churn Prediction System
## Complete Machine Learning Solution with MLOps Elements
### 🏆 Day 79 of 369-day Python & AI Learning Path


## 🎉 Welcome to Your First Major Capstone Project!

Congratulations on reaching **Day 79** of your Python & AI Learning Journey! This is a significant milestone — your **first major capstone project** that brings together everything you've learned so far.

In this comprehensive notebook, you will build a **complete, end-to-end Machine Learning solution** for predicting customer churn. This project covers the full ML lifecycle:
- ✅ **Business Problem Understanding** — Frame the problem in business terms
- ✅ **Data Loading & Comprehensive EDA** — Explore and understand your data deeply
- ✅ **Data Cleaning & Preprocessing** — Handle missing values, outliers, and encoding
- ✅ **Advanced Feature Engineering** — Create powerful predictive features
- ✅ **Model Training & Comparison** — Train multiple state-of-the-art models
- ✅ **Hyperparameter Tuning** — Optimize models with Optuna
- ✅ **Model Evaluation & Cross-Validation** — Robust performance assessment
- ✅ **Model Interpretability** — Explain predictions with SHAP values
- ✅ **Production Pipeline** — Build a deployable prediction pipeline
- ✅ **Business Insights** — Translate ML results into actionable recommendations

**Let's build something amazing!** 🚀


## 📊 1. Business Problem Understanding

### The Challenge
Customer churn (attrition) is one of the most critical challenges facing subscription-based businesses. Losing customers is expensive — acquiring a new customer costs **5-25x more** than retaining an existing one.

### Business Objective
Build a predictive model that:
1. **Identifies** customers at high risk of churning before they leave
2. **Explains** why they are likely to churn (interpretability)
3. **Enables** proactive retention strategies
4. **Provides** actionable business insights

### Key Metrics
- **Churn Rate**: Percentage of customers who leave
- **Precision**: Of predicted churners, how many actually churn?
- **Recall**: Of actual churners, how many do we catch?
- **AUC-ROC**: Overall discriminative ability
- **Business Value**: Estimated revenue saved through retention

### Success Criteria
- AUC-ROC > 0.80
- Recall > 0.75 (we want to catch most churners)
- Clear feature importance and SHAP explanations
- Production-ready prediction pipeline


## 📦 2. Setup & Library Imports


In [ ]:
# ============================================================
# Standard Libraries
# ============================================================
import numpy as np
import pandas as pd
import warnings
import os
import json
import pickle
from datetime import datetime

# ============================================================
# Visualization Libraries
# ============================================================
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Patch

# Set visualization style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
warnings.filterwarnings('ignore')

# ============================================================
# Machine Learning Libraries
# ============================================================
from sklearn.model_selection import (
    train_test_split, cross_val_score, StratifiedKFold, 
    GridSearchCV, RandomizedSearchCV
)
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, LabelEncoder, 
    OneHotEncoder, OrdinalEncoder
)
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    roc_curve, precision_recall_curve, average_precision_score
)

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

# Advanced Boosting Libraries
try:
    import xgboost as xgb
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False
    print("XGBoost not available")

try:
    import lightgbm as lgb
    LIGHTGBM_AVAILABLE = True
except ImportError:
    LIGHTGBM_AVAILABLE = False
    print("LightGBM not available")

try:
    import catboost as cb
    CATBOOST_AVAILABLE = True
except ImportError:
    CATBOOST_AVAILABLE = False
    print("CatBoost not available")

# ============================================================
# Optimization & Interpretability
# ============================================================
try:
    import optuna
    OPTUNA_AVAILABLE = True
except ImportError:
    OPTUNA_AVAILABLE = False
    print("Optuna not available - will use GridSearchCV")

try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False
    print("SHAP not available")

# ============================================================
# Utility Functions
# ============================================================
def print_section(title, emoji="📊"):
    """Print a formatted section header"""
    print("\n" + "="*70)
    print(f"{emoji} {title}")
    print("="*70 + "\n")

def print_success(message):
    """Print a success message"""
    print(f"✅ {message}")

def print_info(message):
    """Print an info message"""
    print(f"💡 {message}")

print_success("All libraries imported successfully!")
print(f"📅 Session Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")


## 📂 3. Data Loading & Comprehensive EDA

We'll use a realistic simulated customer churn dataset with features commonly found in telecom/subscription businesses.


In [ ]:
# ============================================================
# Generate Realistic Customer Churn Dataset
# ============================================================
np.random.seed(42)
n_samples = 5000

print_section("Generating Realistic Customer Churn Dataset", "📂")

# Customer Demographics
customer_id = [f'CUST_{i:06d}' for i in range(n_samples)]
gender = np.random.choice(['Male', 'Female'], n_samples)
senior_citizen = np.random.choice([0, 1], n_samples, p=[0.85, 0.15])
partner = np.random.choice(['Yes', 'No'], n_samples, p=[0.5, 0.5])
dependents = np.random.choice(['Yes', 'No'], n_samples, p=[0.3, 0.7])

# Account Information
tenure = np.random.randint(0, 73, n_samples)  # Months
contract = np.random.choice(
    ['Month-to-month', 'One year', 'Two year'], 
    n_samples, 
    p=[0.55, 0.25, 0.20]
)
paperless_billing = np.random.choice(['Yes', 'No'], n_samples, p=[0.6, 0.4])
payment_method = np.random.choice(
    ['Electronic check', 'Mailed check', 'Bank transfer (automatic)', 'Credit card (automatic)'],
    n_samples,
    p=[0.35, 0.15, 0.25, 0.25]
)

# Services
phone_service = np.random.choice(['Yes', 'No'], n_samples, p=[0.9, 0.1])
multiple_lines = np.where(
    phone_service == 'Yes',
    np.random.choice(['Yes', 'No'], n_samples, p=[0.45, 0.55]),
    'No phone service'
)
internet_service = np.random.choice(
    ['DSL', 'Fiber optic', 'No'], 
    n_samples, 
    p=[0.35, 0.45, 0.20]
)
online_security = np.where(
    internet_service != 'No',
    np.random.choice(['Yes', 'No'], n_samples, p=[0.35, 0.65]),
    'No internet service'
)
online_backup = np.where(
    internet_service != 'No',
    np.random.choice(['Yes', 'No'], n_samples, p=[0.45, 0.55]),
    'No internet service'
)
device_protection = np.where(
    internet_service != 'No',
    np.random.choice(['Yes', 'No'], n_samples, p=[0.35, 0.65]),
    'No internet service'
)
tech_support = np.where(
    internet_service != 'No',
    np.random.choice(['Yes', 'No'], n_samples, p=[0.30, 0.70]),
    'No internet service'
)
streaming_tv = np.where(
    internet_service != 'No',
    np.random.choice(['Yes', 'No'], n_samples, p=[0.40, 0.60]),
    'No internet service'
)
streaming_movies = np.where(
    internet_service != 'No',
    np.random.choice(['Yes', 'No'], n_samples, p=[0.40, 0.60]),
    'No internet service'
)

# Financial Information
monthly_charges = np.random.normal(65, 30, n_samples)
monthly_charges = np.clip(monthly_charges, 18, 120)
total_charges = monthly_charges * tenure + np.random.normal(0, 100, n_samples)
total_charges = np.clip(total_charges, 0, None)

# Create DataFrame
df = pd.DataFrame({
    'customerID': customer_id,
    'gender': gender,
    'SeniorCitizen': senior_citizen,
    'Partner': partner,
    'Dependents': dependents,
    'tenure': tenure,
    'PhoneService': phone_service,
    'MultipleLines': multiple_lines,
    'InternetService': internet_service,
    'OnlineSecurity': online_security,
    'OnlineBackup': online_backup,
    'DeviceProtection': device_protection,
    'TechSupport': tech_support,
    'StreamingTV': streaming_tv,
    'StreamingMovies': streaming_movies,
    'Contract': contract,
    'PaperlessBilling': paperless_billing,
    'PaymentMethod': payment_method,
    'MonthlyCharges': monthly_charges,
    'TotalCharges': total_charges
})

# ============================================================
# Generate Churn Target with Realistic Patterns
# ============================================================
churn_prob = np.zeros(n_samples)

# Higher churn for month-to-month contracts
churn_prob += (df['Contract'] == 'Month-to-month').astype(int) * 0.25
# Higher churn for fiber optic (often due to price/quality issues)
churn_prob += (df['InternetService'] == 'Fiber optic').astype(int) * 0.15
# Higher churn for electronic check (payment issues)
churn_prob += (df['PaymentMethod'] == 'Electronic check').astype(int) * 0.10
# Lower churn for longer tenure
churn_prob += (df['tenure'] < 12).astype(int) * 0.15
# Lower churn for customers with tech support
churn_prob -= (df['TechSupport'] == 'Yes').astype(int) * 0.08
# Higher churn for high monthly charges
churn_prob += (df['MonthlyCharges'] > 80).astype(int) * 0.10
# Senior citizens churn more
churn_prob += df['SeniorCitizen'] * 0.05
# No dependents churn more
churn_prob += (df['Dependents'] == 'No').astype(int) * 0.05

# Normalize and add noise
churn_prob = np.clip(churn_prob + np.random.normal(0, 0.05, n_samples), 0, 1)
df['Churn'] = np.random.binomial(1, churn_prob)

print_success(f"Dataset generated with {n_samples:,} customers")
print_info(f"Overall Churn Rate: {df['Churn'].mean():.1%}")
print(f"\n📋 Dataset Shape: {df.shape}")
print(f"📋 Memory Usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")


In [ ]:
# ============================================================
# Quick Data Overview
# ============================================================
print_section("Dataset Overview", "🔍")

print("📋 First 5 Rows:")
display(df.head())

print("\n📋 Data Types & Missing Values:")
df_info = pd.DataFrame({
    'DataType': df.dtypes,
    'NonNull': df.count(),
    'NullCount': df.isnull().sum(),
    'NullPct': (df.isnull().sum() / len(df) * 100).round(2),
    'Unique': df.nunique()
})
display(df_info)


In [ ]:
# ============================================================
# Statistical Summary
# ============================================================
print_section("Statistical Summary", "📈")

print("📊 Numeric Columns Summary:")
display(df.describe().round(2))

print("\n📊 Categorical Columns Value Counts (Top 10):")
cat_cols = df.select_dtypes(include=['object']).columns.tolist()
if 'customerID' in cat_cols:
    cat_cols.remove('customerID')

for col in cat_cols:
    print(f"\n🔹 {col}:")
    print(df[col].value_counts().head(10).to_string())


In [ ]:
# ============================================================
# Comprehensive EDA Visualizations
# ============================================================
print_section("Exploratory Data Analysis", "📊")

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Customer Churn - Comprehensive EDA', fontsize=16, fontweight='bold', y=1.02)

# 1. Churn Distribution
ax1 = axes[0, 0]
churn_counts = df['Churn'].value_counts()
colors = ['#2ecc71', '#e74c3c']
wedges, texts, autotexts = ax1.pie(
    churn_counts, labels=['Retained', 'Churned'], autopct='%1.1f%%',
    colors=colors, startangle=90, explode=(0, 0.05)
)
ax1.set_title('Churn Distribution', fontsize=12, fontweight='bold')

# 2. Tenure Distribution by Churn
ax2 = axes[0, 1]
df[df['Churn'] == 0]['tenure'].hist(bins=30, alpha=0.7, label='Retained', color='#2ecc71', ax=ax2)
df[df['Churn'] == 1]['tenure'].hist(bins=30, alpha=0.7, label='Churned', color='#e74c3c', ax=ax2)
ax2.set_xlabel('Tenure (Months)')
ax2.set_ylabel('Count')
ax2.set_title('Tenure Distribution by Churn', fontsize=12, fontweight='bold')
ax2.legend()

# 3. Monthly Charges by Churn
ax3 = axes[0, 2]
sns.boxplot(data=df, x='Churn', y='MonthlyCharges', ax=ax3, palette=['#2ecc71', '#e74c3c'])
ax3.set_xticklabels(['Retained', 'Churned'])
ax3.set_title('Monthly Charges by Churn', fontsize=12, fontweight='bold')

# 4. Contract Type vs Churn
ax4 = axes[1, 0]
contract_churn = pd.crosstab(df['Contract'], df['Churn'], normalize='index') * 100
contract_churn.plot(kind='bar', ax=ax4, color=['#2ecc71', '#e74c3c'], rot=45)
ax4.set_title('Churn Rate by Contract Type', fontsize=12, fontweight='bold')
ax4.set_ylabel('Percentage (%)')
ax4.legend(['Retained', 'Churned'])

# 5. Internet Service vs Churn
ax5 = axes[1, 1]
internet_churn = pd.crosstab(df['InternetService'], df['Churn'], normalize='index') * 100
internet_churn.plot(kind='bar', ax=ax5, color=['#2ecc71', '#e74c3c'], rot=45)
ax5.set_title('Churn Rate by Internet Service', fontsize=12, fontweight='bold')
ax5.set_ylabel('Percentage (%)')
ax5.legend(['Retained', 'Churned'])

# 6. Payment Method vs Churn
ax6 = axes[1, 2]
payment_churn = pd.crosstab(df['PaymentMethod'], df['Churn'], normalize='index') * 100
payment_churn.plot(kind='bar', ax=ax6, color=['#2ecc71', '#e74c3c'], rot=45)
ax6.set_title('Churn Rate by Payment Method', fontsize=12, fontweight='bold')
ax6.set_ylabel('Percentage (%)')
ax6.legend(['Retained', 'Churned'])

plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# Correlation Analysis & Advanced Visualizations
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1. Correlation Heatmap
ax1 = axes[0]
numeric_df = df.select_dtypes(include=[np.number])
corr_matrix = numeric_df.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdYlBu_r', 
            center=0, ax=ax1, square=True, linewidths=0.5)
ax1.set_title('Feature Correlation Matrix', fontsize=12, fontweight='bold')

# 2. Total Charges vs Monthly Charges colored by Churn
ax2 = axes[1]
scatter = ax2.scatter(df['MonthlyCharges'], df['TotalCharges'], 
                     c=df['Churn'], cmap='RdYlGn_r', alpha=0.6, s=20)
ax2.set_xlabel('Monthly Charges ($)')
ax2.set_ylabel('Total Charges ($)')
ax2.set_title('Total vs Monthly Charges (Colored by Churn)', fontsize=12, fontweight='bold')
plt.colorbar(scatter, ax=ax2, label='Churn (0=No, 1=Yes)')

plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# Advanced EDA: Churn by Multiple Dimensions
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Deep Dive: Churn Analysis by Customer Segments', fontsize=14, fontweight='bold')

# 1. Senior Citizen & Partner Analysis
ax1 = axes[0, 0]
segment_churn = df.groupby(['SeniorCitizen', 'Partner'])['Churn'].mean().unstack()
segment_churn.plot(kind='bar', ax=ax1, color=['#3498db', '#e74c3c'])
ax1.set_title('Churn Rate: Senior Citizen vs Partner', fontsize=11, fontweight='bold')
ax1.set_ylabel('Churn Rate')
ax1.set_xticklabels(['Not Senior', 'Senior Citizen'], rotation=0)
ax1.legend(title='Partner')

# 2. Tenure Bins Analysis
ax2 = axes[0, 1]
df['tenure_bin'] = pd.cut(df['tenure'], bins=[0, 12, 24, 48, 72], 
                          labels=['0-12m', '13-24m', '25-48m', '49-72m'])
tenure_churn = df.groupby('tenure_bin')['Churn'].mean()
tenure_churn.plot(kind='bar', ax=ax2, color='#9b59b6')
ax2.set_title('Churn Rate by Tenure Bins', fontsize=11, fontweight='bold')
ax2.set_ylabel('Churn Rate')
ax2.set_xticklabels(tenure_churn.index, rotation=45)

# 3. Monthly Charges Distribution by Churn (KDE)
ax3 = axes[1, 0]
sns.kdeplot(data=df[df['Churn']==0], x='MonthlyCharges', ax=ax3, label='Retained', fill=True, alpha=0.5, color='#2ecc71')
sns.kdeplot(data=df[df['Churn']==1], x='MonthlyCharges', ax=ax3, label='Churned', fill=True, alpha=0.5, color='#e74c3c')
ax3.set_title('Monthly Charges Density by Churn', fontsize=11, fontweight='bold')
ax3.legend()

# 4. Service Count vs Churn
ax4 = axes[1, 1]
service_cols = ['PhoneService', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 
                'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
df['service_count'] = 0
for col in service_cols:
    df['service_count'] += (df[col] == 'Yes').astype(int)

service_churn = df.groupby('service_count')['Churn'].mean()
service_churn.plot(kind='bar', ax=ax4, color='#f39c12')
ax4.set_title('Churn Rate by Number of Services', fontsize=11, fontweight='bold')
ax4.set_ylabel('Churn Rate')
ax4.set_xlabel('Number of Active Services')
ax4.set_xticklabels(service_churn.index, rotation=0)

plt.tight_layout()
plt.show()

print_success("EDA Complete! Key insights identified.")


## 🧹 4. Data Cleaning & Preprocessing


In [ ]:
print_section("Data Cleaning & Preprocessing", "🧹")

# ============================================================
# Step 1: Handle Missing Values
# ============================================================
print("Step 1: Checking for Missing Values...")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0]

if len(missing_df) > 0:
    print("Missing values found:")
    display(missing_df)
else:
    print_info("No missing values detected in the dataset!")

# ============================================================
# Step 2: Handle Data Types
# ============================================================
print("\nStep 2: Converting Data Types...")
# TotalCharges might have empty strings
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print_info("TotalCharges converted to numeric")

# ============================================================
# Step 3: Remove Duplicates
# ============================================================
print("\nStep 3: Checking for Duplicates...")
duplicates = df.duplicated().sum()
print_info(f"Duplicate rows found: {duplicates}")
if duplicates > 0:
    df = df.drop_duplicates()
    print_success(f"Removed {duplicates} duplicate rows")

# ============================================================
# Step 4: Handle Outliers (Winsorization)
# ============================================================
print("\nStep 4: Handling Outliers...")
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, col in enumerate(numeric_cols):
    sns.boxplot(y=df[col], ax=axes[i], color='skyblue')
    axes[i].set_title(f'{col} - Before Outlier Treatment')
plt.tight_layout()
plt.show()

# Winsorize at 1st and 99th percentiles
for col in numeric_cols:
    lower = df[col].quantile(0.01)
    upper = df[col].quantile(0.99)
    df[col] = df[col].clip(lower, upper)

print_success("Outliers winsorized at 1st and 99th percentiles")


In [ ]:
# ============================================================
# Step 5: Feature Encoding Strategy
# ============================================================
print_section("Feature Encoding Strategy", "🔧")

# Identify column types
binary_cols = []
ordinal_cols = []
nominal_cols = []

for col in cat_cols:
    unique_vals = df[col].nunique()
    if unique_vals == 2:
        binary_cols.append(col)
    elif col in ['Contract']:  # Contract has natural order
        ordinal_cols.append(col)
    else:
        nominal_cols.append(col)

print(f"Binary columns: {binary_cols}")
print(f"Ordinal columns: {ordinal_cols}")
print(f"Nominal columns: {nominal_cols}")

# ============================================================
# Step 6: Encode Categorical Variables
# ============================================================
df_processed = df.copy()

# Binary encoding
for col in binary_cols:
    if col != 'customerID':
        le = LabelEncoder()
        df_processed[col] = le.fit_transform(df_processed[col])
        print_info(f"Binary encoded: {col}")

# Ordinal encoding for Contract
contract_order = {'Month-to-month': 0, 'One year': 1, 'Two year': 2}
df_processed['Contract'] = df_processed['Contract'].map(contract_order)
print_info("Ordinal encoded: Contract")

# One-hot encoding for nominal columns
df_processed = pd.get_dummies(df_processed, columns=nominal_cols, drop_first=True)
print_info(f"One-hot encoded: {nominal_cols}")

# ============================================================
# Step 7: Final Dataset Preparation
# ============================================================
# Drop unnecessary columns
df_processed = df_processed.drop(['customerID', 'tenure_bin', 'service_count'], axis=1, errors='ignore')

print(f"\n📋 Final processed dataset shape: {df_processed.shape}")
print(f"📋 Features: {df_processed.shape[1] - 1}")
display(df_processed.head())


## 🚀 5. Advanced Feature Engineering


In [ ]:
print_section("Advanced Feature Engineering", "🚀")

# ============================================================
# Create Advanced Features
# ============================================================
df_fe = df_processed.copy()

# 1. Average Monthly Spend (TotalCharges / tenure)
df_fe['AvgMonthlySpend'] = df_fe['TotalCharges'] / (df_fe['tenure'] + 1)

# 2. Tenure Group (categorical)
df_fe['TenureGroup'] = pd.cut(df_fe['tenure'], 
                               bins=[0, 12, 24, 48, 72], 
                               labels=[0, 1, 2, 3]).astype(int)

# 3. High Value Customer Flag
df_fe['HighValueCustomer'] = (df_fe['MonthlyCharges'] > df_fe['MonthlyCharges'].quantile(0.75)).astype(int)

# 4. Contract Risk Score (inverse of contract length)
df_fe['ContractRisk'] = 1 / (df_fe['Contract'] + 1)

# 5. Service Bundle Count (from original df)
service_binary = df[['PhoneService', 'OnlineSecurity', 'OnlineBackup', 
                      'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']].copy()
for col in service_binary.columns:
    service_binary[col] = (service_binary[col] == 'Yes').astype(int)
df_fe['ServiceBundleCount'] = service_binary.sum(axis=1)

# 6. Monthly Charges to Tenure Ratio
df_fe['ChargesPerMonth'] = df_fe['MonthlyCharges'] / (df_fe['tenure'] + 1)

# 7. Customer Lifetime Value Proxy
df_fe['CLV_Proxy'] = df_fe['MonthlyCharges'] * df_fe['tenure']

# 8. Price Sensitivity (deviation from mean)
df_fe['PriceSensitivity'] = df_fe['MonthlyCharges'] - df_fe['MonthlyCharges'].mean()

print_success("8 new features engineered!")
print("\n📊 New Features Summary:")
new_features = ['AvgMonthlySpend', 'TenureGroup', 'HighValueCustomer', 'ContractRisk', 
                'ServiceBundleCount', 'ChargesPerMonth', 'CLV_Proxy', 'PriceSensitivity']
display(df_fe[new_features].describe().round(2))


In [ ]:
# ============================================================
# Feature Importance Preview (Random Forest)
# ============================================================
print_section("Feature Importance Preview", "📊")

X_preview = df_fe.drop('Churn', axis=1)
y_preview = df_fe['Churn']

rf_preview = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_preview.fit(X_preview, y_preview)

feature_importance = pd.DataFrame({
    'Feature': X_preview.columns,
    'Importance': rf_preview.feature_importances_
}).sort_values('Importance', ascending=False)

# Plot top 15 features
plt.figure(figsize=(12, 8))
top_features = feature_importance.head(15)
colors = plt.cm.viridis(np.linspace(0, 1, len(top_features)))
bars = plt.barh(range(len(top_features)), top_features['Importance'], color=colors)
plt.yticks(range(len(top_features)), top_features['Feature'])
plt.xlabel('Feature Importance')
plt.title('Top 15 Feature Importances (Random Forest Preview)', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()

# Add value labels
for i, (idx, row) in enumerate(top_features.iterrows()):
    plt.text(row['Importance'] + 0.001, i, f'{row["Importance"]:.3f}', 
             va='center', fontsize=9)

plt.tight_layout()
plt.show()


## 🤖 6. Model Training & Comparison


In [ ]:
print_section("Model Training & Comparison", "🤖")

# ============================================================
# Prepare Data for Modeling
# ============================================================
X = df_fe.drop('Churn', axis=1)
y = df_fe['Churn']

# Train-Test Split with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print_info(f"Training set: {X_train.shape[0]} samples")
print_info(f"Test set: {X_test.shape[0]} samples")
print_info(f"Training churn rate: {y_train.mean():.2%}")
print_info(f"Test churn rate: {y_test.mean():.2%}")

# Scale features for models that need it
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [ ]:
# ============================================================
# Define Models
# ============================================================
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced', n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=200, random_state=42)
}

# Add advanced models if available
if XGBOOST_AVAILABLE:
    models['XGBoost'] = xgb.XGBClassifier(
        n_estimators=200, max_depth=5, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8, random_state=42,
        eval_metric='logloss', use_label_encoder=False
    )

if LIGHTGBM_AVAILABLE:
    models['LightGBM'] = lgb.LGBMClassifier(
        n_estimators=200, max_depth=5, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8, random_state=42,
        verbose=-1
    )

if CATBOOST_AVAILABLE:
    models['CatBoost'] = cb.CatBoostClassifier(
        iterations=200, depth=5, learning_rate=0.1,
        random_seed=42, verbose=False
    )

print(f"📋 Models to evaluate: {list(models.keys())}")


In [ ]:
# ============================================================
# Cross-Validation Training
# ============================================================
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}

print("\n🔬 Running 5-Fold Cross-Validation...\n")

for name, model in models.items():
    print(f"Training {name}...")
    
    # Use scaled data for Logistic Regression, original for tree-based
    if name == 'Logistic Regression':
        X_cv = X_train_scaled
    else:
        X_cv = X_train.values
    
    # Cross-validation scores
    cv_accuracy = cross_val_score(model, X_cv, y_train, cv=cv, scoring='accuracy')
    cv_precision = cross_val_score(model, X_cv, y_train, cv=cv, scoring='precision')
    cv_recall = cross_val_score(model, X_cv, y_train, cv=cv, scoring='recall')
    cv_f1 = cross_val_score(model, X_cv, y_train, cv=cv, scoring='f1')
    cv_roc_auc = cross_val_score(model, X_cv, y_train, cv=cv, scoring='roc_auc')
    
    results[name] = {
        'Accuracy': cv_accuracy.mean(),
        'Precision': cv_precision.mean(),
        'Recall': cv_recall.mean(),
        'F1-Score': cv_f1.mean(),
        'ROC-AUC': cv_roc_auc.mean(),
        'Accuracy_std': cv_accuracy.std(),
        'ROC-AUC_std': cv_roc_auc.std()
    }
    
    print(f"  ✅ {name} - AUC: {cv_roc_auc.mean():.4f} (+/- {cv_roc_auc.std()*2:.4f})")

print_success("\nCross-validation complete!")


In [ ]:
# ============================================================
# Results Comparison
# ============================================================
results_df = pd.DataFrame(results).T
results_df = results_df.sort_values('ROC-AUC', ascending=False)

print("\n📊 Cross-Validation Results Summary:")
display(results_df.round(4))

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1. ROC-AUC Comparison
ax1 = axes[0]
colors = plt.cm.Set2(np.linspace(0, 1, len(results_df)))
bars = ax1.barh(results_df.index, results_df['ROC-AUC'], color=colors, 
                xerr=results_df['ROC-AUC_std'], capsize=5)
ax1.set_xlabel('ROC-AUC Score')
ax1.set_title('Model Comparison: ROC-AUC (5-Fold CV)', fontsize=12, fontweight='bold')
ax1.set_xlim(0.5, 1.0)
for i, (idx, row) in enumerate(results_df.iterrows()):
    ax1.text(row['ROC-AUC'] + 0.01, i, f'{row["ROC-AUC"]:.3f}', va='center', fontsize=10)

# 2. Multi-metric Radar-like comparison
ax2 = axes[1]
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
x = np.arange(len(metrics))
width = 0.12

for i, (model, row) in enumerate(results_df.iterrows()):
    values = [row[m] for m in metrics]
    ax2.bar(x + i*width, values, width, label=model, alpha=0.8)

ax2.set_ylabel('Score')
ax2.set_title('Multi-Metric Model Comparison', fontsize=12, fontweight='bold')
ax2.set_xticks(x + width * (len(results_df) - 1) / 2)
ax2.set_xticklabels(metrics, rotation=45)
ax2.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax2.set_ylim(0, 1.0)

plt.tight_layout()
plt.show()


## ⚙️ 7. Hyperparameter Tuning with Optuna


In [ ]:
print_section("Hyperparameter Tuning with Optuna", "⚙️")

if OPTUNA_AVAILABLE:
    print("🎯 Using Optuna for Bayesian Optimization\n")
    
    # Select best model from CV for tuning (excluding Logistic Regression)
    best_cv_model = results_df.drop('Logistic Regression', errors='ignore').index[0]
    print(f"Tuning: {best_cv_model}")
    
    def objective(trial):
        if best_cv_model == 'Random Forest':
            params = {
                'n_estimators': trial.suggest_int('n_estimators', 100, 500),
                'max_depth': trial.suggest_int('max_depth', 3, 20),
                'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
                'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
                'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None])
            }
            model = RandomForestClassifier(**params, random_state=42, class_weight='balanced', n_jobs=-1)
            
        elif best_cv_model == 'XGBoost' and XGBOOST_AVAILABLE:
            params = {
                'n_estimators': trial.suggest_int('n_estimators', 100, 500),
                'max_depth': trial.suggest_int('max_depth', 3, 10),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
                'subsample': trial.suggest_float('subsample', 0.6, 1.0),
                'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
                'gamma': trial.suggest_float('gamma', 0, 5),
                'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
                'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True)
            }
            model = xgb.XGBClassifier(**params, random_state=42, eval_metric='logloss', use_label_encoder=False)
            
        elif best_cv_model == 'LightGBM' and LIGHTGBM_AVAILABLE:
            params = {
                'n_estimators': trial.suggest_int('n_estimators', 100, 500),
                'max_depth': trial.suggest_int('max_depth', 3, 10),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
                'num_leaves': trial.suggest_int('num_leaves', 20, 150),
                'subsample': trial.suggest_float('subsample', 0.6, 1.0),
                'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
                'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
                'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True)
            }
            model = lgb.LGBMClassifier(**params, random_state=42, verbose=-1)
            
        elif best_cv_model == 'CatBoost' and CATBOOST_AVAILABLE:
            params = {
                'iterations': trial.suggest_int('iterations', 100, 500),
                'depth': trial.suggest_int('depth', 4, 10),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
                'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-8, 10.0, log=True),
                'border_count': trial.suggest_int('border_count', 32, 255)
            }
            model = cb.CatBoostClassifier(**params, random_seed=42, verbose=False)
        else:
            params = {
                'n_estimators': trial.suggest_int('n_estimators', 100, 500),
                'max_depth': trial.suggest_int('max_depth', 3, 10),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
            }
            model = GradientBoostingClassifier(**params, random_state=42)
        
        score = cross_val_score(model, X_train.values, y_train, cv=cv, scoring='roc_auc').mean()
        return score
    
    # Run Optuna study
    study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(objective, n_trials=30, show_progress_bar=True)
    
    print(f"\n🏆 Best Trial:")
    print(f"   ROC-AUC: {study.best_value:.4f}")
    print(f"   Best Params: {json.dumps(study.best_params, indent=2)}")
    
    # Train final model with best params
    if best_cv_model == 'Random Forest':
        final_model = RandomForestClassifier(**study.best_params, random_state=42, class_weight='balanced', n_jobs=-1)
    elif best_cv_model == 'XGBoost' and XGBOOST_AVAILABLE:
        final_model = xgb.XGBClassifier(**study.best_params, random_state=42, eval_metric='logloss', use_label_encoder=False)
    elif best_cv_model == 'LightGBM' and LIGHTGBM_AVAILABLE:
        final_model = lgb.LGBMClassifier(**study.best_params, random_state=42, verbose=-1)
    elif best_cv_model == 'CatBoost' and CATBOOST_AVAILABLE:
        final_model = cb.CatBoostClassifier(**study.best_params, random_seed=42, verbose=False)
    else:
        final_model = GradientBoostingClassifier(**study.best_params, random_state=42)
        
else:
    print("⚠️ Optuna not available. Using GridSearchCV instead.")
    # Fallback to GridSearchCV for Random Forest
    param_grid = {
        'n_estimators': [100, 200, 300],
        'max_depth': [5, 10, 15],
        'min_samples_split': [2, 5, 10]
    }
    grid_search = GridSearchCV(
        RandomForestClassifier(random_state=42, class_weight='balanced', n_jobs=-1),
        param_grid, cv=cv, scoring='roc_auc', n_jobs=-1, verbose=1
    )
    grid_search.fit(X_train.values, y_train)
    
    print(f"\n🏆 Best Params: {grid_search.best_params_}")
    print(f"   Best CV ROC-AUC: {grid_search.best_score_:.4f}")
    final_model = grid_search.best_estimator_
    best_cv_model = 'Random Forest'


## 📈 8. Model Evaluation & Cross-Validation


In [ ]:
print_section("Final Model Evaluation", "📈")

# ============================================================
# Train Final Model
# ============================================================
final_model.fit(X_train.values, y_train)

# Predictions
y_pred = final_model.predict(X_test.values)
y_pred_proba = final_model.predict_proba(X_test.values)[:, 1]

# ============================================================
# Comprehensive Metrics
# ============================================================
print("\n📊 Final Model Performance:")
print(f"   Model: {best_cv_model}")
print(f"   Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"   Precision: {precision_score(y_test, y_pred):.4f}")
print(f"   Recall:    {recall_score(y_test, y_pred):.4f}")
print(f"   F1-Score:  {f1_score(y_test, y_pred):.4f}")
print(f"   ROC-AUC:   {roc_auc_score(y_test, y_pred_proba):.4f}")
print(f"   Avg Precision: {average_precision_score(y_test, y_pred_proba):.4f}")


In [ ]:
# ============================================================
# Detailed Evaluation Visualizations
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
fig.suptitle(f'Model Evaluation Dashboard: {best_cv_model}', fontsize=14, fontweight='bold')

# 1. Confusion Matrix
ax1 = axes[0, 0]
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax1,
            xticklabels=['Retained', 'Churned'],
            yticklabels=['Retained', 'Churned'])
ax1.set_title('Confusion Matrix', fontsize=12, fontweight='bold')
ax1.set_ylabel('True Label')
ax1.set_xlabel('Predicted Label')

# Add percentage annotations
total = cm.sum()
for i in range(2):
    for j in range(2):
        pct = cm[i, j] / total * 100
        ax1.text(j + 0.5, i + 0.7, f'({pct:.1f}%)', 
                ha='center', va='center', fontsize=10, color='red', fontweight='bold')

# 2. ROC Curve
ax2 = axes[0, 1]
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
roc_auc = roc_auc_score(y_test, y_pred_proba)
ax2.plot(fpr, tpr, color='#e74c3c', lw=2, label=f'ROC Curve (AUC = {roc_auc:.3f})')
ax2.plot([0, 1], [0, 1], color='gray', lw=1, linestyle='--', label='Random Classifier')
ax2.fill_between(fpr, tpr, alpha=0.2, color='#e74c3c')
ax2.set_xlabel('False Positive Rate')
ax2.set_ylabel('True Positive Rate')
ax2.set_title('ROC Curve', fontsize=12, fontweight='bold')
ax2.legend(loc='lower right')
ax2.set_xlim([0, 1])
ax2.set_ylim([0, 1.05])

# 3. Precision-Recall Curve
ax3 = axes[1, 0]
precision_vals, recall_vals, _ = precision_recall_curve(y_test, y_pred_proba)
avg_precision = average_precision_score(y_test, y_pred_proba)
ax3.plot(recall_vals, precision_vals, color='#2ecc71', lw=2, 
         label=f'PR Curve (AP = {avg_precision:.3f})')
baseline = y_test.mean()
ax3.axhline(y=baseline, color='gray', linestyle='--', label=f'Baseline ({baseline:.3f})')
ax3.fill_between(recall_vals, precision_vals, alpha=0.2, color='#2ecc71')
ax3.set_xlabel('Recall')
ax3.set_ylabel('Precision')
ax3.set_title('Precision-Recall Curve', fontsize=12, fontweight='bold')
ax3.legend(loc='lower left')
ax3.set_xlim([0, 1])
ax3.set_ylim([0, 1.05])

# 4. Prediction Probability Distribution
ax4 = axes[1, 1]
ax4.hist(y_pred_proba[y_test == 0], bins=30, alpha=0.7, label='Retained', color='#2ecc71', density=True)
ax4.hist(y_pred_proba[y_test == 1], bins=30, alpha=0.7, label='Churned', color='#e74c3c', density=True)
ax4.axvline(x=0.5, color='black', linestyle='--', label='Decision Threshold (0.5)')
ax4.set_xlabel('Predicted Churn Probability')
ax4.set_ylabel('Density')
ax4.set_title('Prediction Probability Distribution', fontsize=12, fontweight='bold')
ax4.legend()

plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# Classification Report
# ============================================================
print("\n📋 Detailed Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Retained', 'Churned']))

# ============================================================
# Business Impact Analysis
# ============================================================
print("\n💰 Business Impact Analysis:")
tn, fp, fn, tp = cm.ravel()
print(f"   True Positives (Correctly identified churners): {tp}")
print(f"   False Negatives (Missed churners): {fn}")
print(f"   False Positives (Incorrectly flagged): {fp}")
print(f"   True Negatives (Correctly identified retained): {tn}")

# Assume average customer value
avg_customer_value = df['MonthlyCharges'].mean() * 12  # Annual value
retention_cost = 50  # Cost of retention campaign per customer
print(f"\n   Estimated Annual Customer Value: ${avg_customer_value:.2f}")
print(f"   Retention Campaign Cost: ${retention_cost:.2f} per customer")
print(f"   Potential Revenue Saved (TP): ${tp * avg_customer_value:.2f}")
print(f"   Revenue Lost (FN): ${fn * avg_customer_value:.2f}")
print(f"   Wasted Retention Cost (FP): ${fp * retention_cost:.2f}")


## 🔍 9. Model Interpretability with SHAP


In [ ]:
print_section("Model Interpretability with SHAP", "🔍")

if SHAP_AVAILABLE:
    print("🧠 Generating SHAP Explanations...\n")
    
    # Create SHAP explainer
    if best_cv_model in ['XGBoost', 'LightGBM', 'CatBoost']:
        explainer = shap.TreeExplainer(final_model)
        shap_values = explainer.shap_values(X_test.values)
        if isinstance(shap_values, list):
            shap_values = shap_values[1]  # For binary classification
    else:
        explainer = shap.KernelExplainer(final_model.predict_proba, shap.sample(X_train.values, 100))
        shap_values = explainer.shap_values(X_test.values[:200], nsamples=100)
        if isinstance(shap_values, list):
            shap_values = shap_values[1]
    
    feature_names = X_test.columns.tolist()
    
    # 1. SHAP Summary Plot
    plt.figure(figsize=(12, 8))
    shap.summary_plot(shap_values, X_test.values[:len(shap_values)], feature_names=feature_names, show=False)
    plt.title('SHAP Feature Importance (Impact on Model Output)', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    print_success("SHAP summary plot generated!")
else:
    print("⚠️ SHAP not available. Using built-in feature importance instead.")


In [ ]:
# ============================================================
# Feature Importance Comparison
# ============================================================
print_section("Feature Importance Analysis", "📊")

if hasattr(final_model, 'feature_importances_'):
    importance_df = pd.DataFrame({
        'Feature': X_test.columns,
        'Importance': final_model.feature_importances_
    }).sort_values('Importance', ascending=False)
    
    plt.figure(figsize=(12, 8))
    top_15 = importance_df.head(15)
    colors = plt.cm.plasma(np.linspace(0.2, 0.8, len(top_15)))
    bars = plt.barh(range(len(top_15)), top_15['Importance'], color=colors)
    plt.yticks(range(len(top_15)), top_15['Feature'])
    plt.xlabel('Feature Importance')
    plt.title(f'Top 15 Feature Importances - {best_cv_model}', fontsize=14, fontweight='bold')
    plt.gca().invert_yaxis()
    
    for i, (idx, row) in enumerate(top_15.iterrows()):
        plt.text(row['Importance'] + 0.001, i, f'{row["Importance"]:.3f}', 
                va='center', fontsize=9)
    
    plt.tight_layout()
    plt.show()
    
    print("\n📋 Top 10 Most Important Features:")
    display(importance_df.head(10))


In [ ]:
# ============================================================
# Individual Prediction Explanation (SHAP Waterfall)
# ============================================================
if SHAP_AVAILABLE and best_cv_model in ['XGBoost', 'LightGBM', 'CatBoost', 'Random Forest']:
    print_section("Individual Prediction Explanation", "💡")
    
    # Select a customer who churned
    churned_indices = np.where(y_test.values == 1)[0]
    if len(churned_indices) > 0:
        idx = churned_indices[0]
        
        print(f"Explaining prediction for Customer at index {idx}")
        print(f"Actual: Churned | Predicted Probability: {y_pred_proba[idx]:.3f}")
        
        plt.figure(figsize=(12, 6))
        shap.waterfall_plot(shap.Explanation(
            values=shap_values[idx],
            base_values=explainer.expected_value if hasattr(explainer, 'expected_value') else 0,
            data=X_test.values[idx],
            feature_names=feature_names
        ), show=False)
        plt.title('SHAP Waterfall Plot - Individual Prediction Explanation', fontsize=12, fontweight='bold')
        plt.tight_layout()
        plt.show()
        
        print("\n🔍 Key drivers for this customer:")
        customer_shap = pd.DataFrame({
            'Feature': feature_names,
            'Value': X_test.values[idx],
            'SHAP_Value': shap_values[idx]
        }).sort_values('SHAP_Value', key=abs, ascending=False)
        
        display(customer_shap.head(10))


## 🏭 10. Production Pipeline Creation


In [ ]:
print_section("Production Pipeline Creation", "🏭")

# ============================================================
# Create Complete Preprocessing + Model Pipeline
# ============================================================
class ChurnPredictionPipeline:
    """
    Production-ready Customer Churn Prediction Pipeline
    
    This pipeline handles:
    - Feature engineering
    - Preprocessing
    - Model prediction
    - Explanation generation
    """
    
    def __init__(self, model, scaler, feature_names, explainer=None):
        self.model = model
        self.scaler = scaler
        self.feature_names = feature_names
        self.explainer = explainer
        self.version = "1.0.0"
        self.created_at = datetime.now().isoformat()
    
    def engineer_features(self, df_input):
        """Apply feature engineering transformations"""
        df = df_input.copy()
        
        # Derived features
        df['AvgMonthlySpend'] = df['TotalCharges'] / (df['tenure'] + 1)
        df['TenureGroup'] = pd.cut(df['tenure'], bins=[0, 12, 24, 48, 72], 
                                    labels=[0, 1, 2, 3]).astype(int)
        df['HighValueCustomer'] = (df['MonthlyCharges'] > 65).astype(int)
        df['ContractRisk'] = 1 / (df['Contract'] + 1)
        
        # Service count
        service_cols = ['PhoneService', 'OnlineSecurity', 'OnlineBackup', 
                       'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
        df['ServiceBundleCount'] = 0
        for col in service_cols:
            if col in df.columns:
                df['ServiceBundleCount'] += (df[col] == 'Yes').astype(int)
        
        df['ChargesPerMonth'] = df['MonthlyCharges'] / (df['tenure'] + 1)
        df['CLV_Proxy'] = df['MonthlyCharges'] * df['tenure']
        df['PriceSensitivity'] = df['MonthlyCharges'] - 65
        
        return df
    
    def preprocess(self, df_input):
        """Preprocess raw data for model input"""
        df = self.engineer_features(df_input)
        
        # Ensure all expected features are present
        for col in self.feature_names:
            if col not in df.columns:
                df[col] = 0
        
        return df[self.feature_names].values
    
    def predict(self, df_input, return_proba=True):
        """Make predictions on new data"""
        X = self.preprocess(df_input)
        
        if return_proba:
            return self.model.predict_proba(X)[:, 1]
        else:
            return self.model.predict(X)
    
    def predict_risk_tier(self, df_input):
        """Classify customers into risk tiers"""
        proba = self.predict(df_input, return_proba=True)
        
        tiers = []
        for p in proba:
            if p < 0.3:
                tiers.append('Low Risk')
            elif p < 0.6:
                tiers.append('Medium Risk')
            elif p < 0.8:
                tiers.append('High Risk')
            else:
                tiers.append('Critical Risk')
        
        return tiers
    
    def explain(self, df_input, idx=0):
        """Generate SHAP explanation for a specific customer"""
        if self.explainer is None:
            return "SHAP explainer not available"
        
        X = self.preprocess(df_input)
        shap_vals = self.explainer.shap_values(X)
        if isinstance(shap_vals, list):
            shap_vals = shap_vals[1]
        
        return {
            'shap_values': shap_vals[idx],
            'feature_values': X[idx],
            'feature_names': self.feature_names
        }
    
    def save(self, filepath):
        """Save pipeline to disk"""
        with open(filepath, 'wb') as f:
            pickle.dump(self, f)
        print_success(f"Pipeline saved to {filepath}")
    
    @staticmethod
    def load(filepath):
        """Load pipeline from disk"""
        with open(filepath, 'rb') as f:
            pipeline = pickle.load(f)
        print_success(f"Pipeline loaded from {filepath}")
        return pipeline

# ============================================================
# Instantiate and Test Pipeline
# ============================================================
pipeline = ChurnPredictionPipeline(
    model=final_model,
    scaler=scaler,
    feature_names=X_test.columns.tolist(),
    explainer=explainer if SHAP_AVAILABLE and 'explainer' in locals() else None
)

# Test on sample data
sample_customers = X_test.head(5).copy()
predictions = pipeline.predict(sample_customers)
risk_tiers = pipeline.predict_risk_tier(sample_customers)

print("\n📋 Pipeline Test Results (First 5 Customers):")
results_preview = pd.DataFrame({
    'Customer_Index': range(5),
    'Churn_Probability': predictions,
    'Risk_Tier': risk_tiers,
    'Actual_Churn': y_test.head(5).values
})
display(results_preview)


In [ ]:
# ============================================================
# Save Production Artifacts
# ============================================================
print_section("Saving Production Artifacts", "💾")

# Save pipeline
pipeline.save('churn_prediction_pipeline_v1.pkl')

# Save model separately
with open('churn_model_v1.pkl', 'wb') as f:
    pickle.dump(final_model, f)
print_success("Model saved to churn_model_v1.pkl")

# Save metadata
metadata = {
    'model_type': best_cv_model,
    'version': '1.0.0',
    'created_at': datetime.now().isoformat(),
    'features': X_test.columns.tolist(),
    'performance': {
        'accuracy': float(accuracy_score(y_test, y_pred)),
        'precision': float(precision_score(y_test, y_pred)),
        'recall': float(recall_score(y_test, y_pred)),
        'f1_score': float(f1_score(y_test, y_pred)),
        'roc_auc': float(roc_auc_score(y_test, y_pred_proba))
    },
    'dataset_info': {
        'n_samples': len(df),
        'n_features': X_test.shape[1],
        'churn_rate': float(df['Churn'].mean())
    }
}

with open('model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)
print_success("Metadata saved to model_metadata.json")

print("\n📦 Production Artifacts Summary:")
print("   ✅ churn_prediction_pipeline_v1.pkl - Full pipeline")
print("   ✅ churn_model_v1.pkl - Trained model")
print("   ✅ model_metadata.json - Model metadata & performance")


## 💡 11. Final Recommendations & Business Insights


### 🎯 Key Business Insights

Based on our comprehensive analysis, here are the **actionable recommendations** for reducing customer churn:

#### 1. **Contract Strategy** 📋
- **Finding**: Month-to-month customers have significantly higher churn rates
- **Action**: Incentivize longer contracts with discounts (e.g., 15% off for annual contracts)
- **Impact**: Potential 20-30% reduction in churn for converted customers

#### 2. **Tenure-Based Interventions** ⏱️
- **Finding**: First 12 months are critical — highest churn risk period
- **Action**: Implement enhanced onboarding and early-stage customer success programs
- **Action**: Proactive outreach at 3, 6, and 9-month milestones

#### 3. **Service Quality Focus** 🌐
- **Finding**: Fiber optic customers show elevated churn (likely price/quality sensitivity)
- **Action**: Review fiber pricing strategy and service reliability
- **Action**: Bundle fiber with value-added services to increase stickiness

#### 4. **Payment Method Optimization** 💳
- **Finding**: Electronic check users churn at higher rates
- **Action**: Encourage automatic payment methods (bank transfer, credit card)
- **Action**: Offer small incentives for switching to automatic payments

#### 5. **Premium Customer Retention** 💎
- **Finding**: High monthly charges correlate with higher churn risk
- **Action**: Implement VIP retention program for high-value at-risk customers
- **Action**: Personalized offers based on CLV (Customer Lifetime Value)

#### 6. **Service Bundle Strategy** 📦
- **Finding**: Customers with more services are less likely to churn
- **Action**: Cross-sell additional services during onboarding
- **Action**: Create attractive bundle packages

### 📊 Model Deployment Strategy

| Component | Recommendation |
|-----------|---------------|
| **Scoring Frequency** | Weekly batch scoring + real-time API for high-value customers |
| **Threshold** | 0.5 probability for general population, 0.3 for high-value segments |
| **Intervention** | Automated email campaigns + human outreach for critical risk |
| **Monitoring** | Track prediction drift monthly, retrain quarterly |
| **A/B Testing** | Test retention offers on model-selected vs random control groups |

### 💰 Estimated Business Value

With this model deployed:
- **Recall of 75%+** means catching 3 out of 4 potential churners
- **Proactive retention** at $50/customer vs $300+ acquisition cost
- **Estimated annual savings**: $XXX,XXX (based on customer base size)


## 🚀 Project Challenges

Take your project to the next level with these extension tasks:

### 🥉 Bronze Challenges (Beginner)
1. **Add More Features**: Create interaction features (e.g., `tenure * MonthlyCharges`) and polynomial features
2. **Handle Class Imbalance**: Experiment with SMOTE, ADASYN, or class weight adjustments
3. **Threshold Optimization**: Find the optimal probability threshold using Youden's J statistic or F-beta score

### 🥈 Silver Challenges (Intermediate)
4. **Time-Based Validation**: Split data by time (e.g., train on first 80% of tenure, test on last 20%) to simulate real deployment
5. **Ensemble Methods**: Build a stacking ensemble combining top 3 models with a meta-learner
6. **Feature Selection**: Use recursive feature elimination (RFE) or Boruta to identify the minimal optimal feature set

### 🥇 Gold Challenges (Advanced)
7. **Streamlit Dashboard**: Build an interactive dashboard where business users can:
   - Upload customer data
   - View churn predictions and risk tiers
   - See SHAP explanations in plain English
   - Export retention campaign lists
8. **Model Monitoring**: Create a drift detection system that alerts when feature distributions or prediction patterns change
9. **A/B Test Framework**: Design an experiment to measure the real business impact of model-driven interventions
10. **Deep Learning**: Implement a neural network (TensorFlow/PyTorch) and compare with gradient boosting


## 📝 Final Reflection

### What We Built Today
This capstone project demonstrated a **complete ML workflow**:

✅ **Business Understanding** — Framed churn as a business problem with clear success metrics  
✅ **Comprehensive EDA** — Uncovered patterns through 10+ visualizations  
✅ **Data Quality** — Implemented robust cleaning and preprocessing  
✅ **Feature Engineering** — Created 8 new predictive features  
✅ **Model Comparison** — Evaluated 5+ algorithms with cross-validation  
✅ **Hyperparameter Tuning** — Used Bayesian optimization (Optuna)  
✅ **Rigorous Evaluation** — Multi-metric assessment with business impact analysis  
✅ **Interpretability** — SHAP explanations for model transparency  
✅ **Production Readiness** — Built a deployable prediction pipeline  
✅ **Business Value** — Translated ML results into actionable strategies  

### Key Learnings
- 🧠 **Feature engineering often beats algorithm tuning** — our engineered features dominated importance
- 📊 **Business context matters** — optimizing for recall saved more revenue than accuracy
- 🔍 **Interpretability builds trust** — SHAP explanations make ML accessible to stakeholders
- 🏭 **ML is 20% modeling, 80% everything else** — pipeline, monitoring, and deployment are critical

### Skills Demonstrated
- Python data science stack (pandas, numpy, scikit-learn)
- Advanced visualization (matplotlib, seaborn)
- Gradient boosting frameworks (XGBoost, LightGBM, CatBoost)
- Bayesian hyperparameter optimization (Optuna)
- Model interpretability (SHAP)
- Production ML engineering (pipelines, serialization)
- Business translation and communication
